https://www.geeksforgeeks.org/machine-learning/recommendation-system-in-python/

Yaptığım düzeltmeler ve sonuç:

1. Eksik importlar eklendi (csr_matrix, NearestNeighbors), ayrıca kurulu olmayan seaborn paketini de kurdum.

2. Veri yolu düzeltildi ve ratings, movies ile merge edilerek title sütunu eklendi (diğer hücreler değişmeden çalışabilsin diye).

3. recommend_similar fonksiyonuna find_movie yardımcı fonksiyonu eklendi — MovieLens başlıkları "Dark Knight, The (2008)" formatında olduğu için "The Dark Knight" gibi doğal aramaları da doğru eşleştiriyor artık. İlk denemede yanlışlıkla alakasız bir filme eşleşiyordu (Batman: The Dark Knight Returns...), bunu da başa-denk-gelme önceliği ekleyerek düzelttim.

*item-based collaborative filtering (kNN + cosine similarity) yaklaşımının çalışan bir örneği*

Adım 1: Kütüphaneleri İçe Aktarın

In [4]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

Adım 2: Veri Kümesinin Yüklenmesi

Veri setimizi yükleyeceğiz,

Sütunlar : userId, movieId, title, rating.

head() : Hızlı bir kontrol için ilk 5 satırı gösterir.

In [5]:
DATA_DIR = r"C:\Users\Fatmanur\Desktop\Staj\dataset\ml-latest-small"

ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")

# ratings tablosunda film adi yok (sadece movieId var), movies ile birlestirip ekliyoruz
ratings = ratings.merge(movies[["movieId", "title"]], on="movieId")
ratings.head()

,userId,movieId,rating,timestamp,title
0,1,1,4.0,964982703,Toy Story (1995)
1,5,1,4.0,847434962,Toy Story (1995)
2,7,1,4.5,1106635946,Toy Story (1995)
3,15,1,2.5,1510577970,Toy Story (1995)
4,17,1,4.5,1305696483,Toy Story (1995)


Adım 3 : Kullanıcı-Öğe Matrisini Oluşturun

Kullanıcı-ürün seyrek matrisini oluşturuyoruz,

user_mapper : Kullanıcı kimliklerini matris indeksine eşler.

movie_mapper : film kimliklerini matris indeksine eşler.

movie_inv_mapper : ters arama (indeks: movieId).

Satırlar = Filmler

Sütunlar = Kullanıcılar

Değerler = Puanlar

In [6]:
def create_matrix(df):
    user_mapper = {uid: i for i, uid in enumerate(df['userId'].unique())}
    movie_mapper = {mid: i for i, mid in enumerate(df['movieId'].unique())}
    movie_inv_mapper = {i: mid for mid, i in movie_mapper.items()}

    user_index = df['userId'].map(user_mapper)
    movie_index = df['movieId'].map(movie_mapper)

    X = csr_matrix((df["rating"], (movie_index, user_index)),
                   shape=(len(movie_mapper), len(user_mapper)))
    return X, movie_mapper, movie_inv_mapper


X, movie_mapper, movie_inv_mapper = create_matrix(ratings)

user_item_matrix = ratings.pivot_table(
    index="title", columns="userId", values="rating")
print(user_item_matrix.iloc[:10, :5])

userId                                    1   2   3   4   5
title                                                      
'71 (2014)                              NaN NaN NaN NaN NaN
'Hellboy': The Seeds of Creation (2004) NaN NaN NaN NaN NaN
'Round Midnight (1986)                  NaN NaN NaN NaN NaN
'Salem's Lot (2004)                     NaN NaN NaN NaN NaN
'Til There Was You (1997)               NaN NaN NaN NaN NaN
'Tis the Season for Love (2015)         NaN NaN NaN NaN NaN
'burbs, The (1989)                      NaN NaN NaN NaN NaN
'night Mother (1986)                    NaN NaN NaN NaN NaN
(500) Days of Summer (2009)             NaN NaN NaN NaN NaN
*batteries not included (1987)          NaN NaN NaN NaN NaN


In [7]:
def find_movie(title_query, df):
    query = title_query.strip()
    # ratings her kullanici-puan satirinda basligi tekrar ediyor, once tekillestir
    candidates = df.drop_duplicates("movieId")[["movieId", "title"]]

    def starts_with(q):
        return candidates[candidates['title'].str.lower().str.startswith(q.lower())]

    # MovieLens basliklari "Baslik, The (yil)" seklinde saklaniyor (orn. "Dark Knight, The (2008)").
    # Once dogrudan basa-denk-gelen eslesme, olmazsa "The X" -> "X, The" donusumuyle tekrar dene.
    result = starts_with(query)
    if result.empty and query.lower().startswith("the "):
        result = starts_with(query[4:] + ", The")
    if not result.empty:
        return result

    # hicbiri basa denk gelmediyse, basligin herhangi bir yerinde gecen eslesmelere bak;
    # en kisa baslik genelde en alakali eslesme olur (uzun/turev basliklari elemek icin)
    contains = candidates[candidates['title'].str.contains(query, case=False, regex=False, na=False)].copy()
    return contains.sort_values(by="title", key=lambda s: s.str.len())

Adım 4: Öneri Fonksiyonunu Tanımlayın

Öneri işlevi için kullanılacak fonksiyonu tanımlamamız gerekiyor.

movie_id : Verilen filmin kimliğini alır.

movie_idx : Matristeki satır indeksini bulur.

NearestNeighbors : Kosinüs benzerliği kullanarak en yakın k filmi bulur .

In [8]:
def recommend_similar(movie_title, df, X, movie_mapper, movie_inv_mapper, k=5):
    matches = find_movie(movie_title, df)
    if matches.empty:
        print(f"'{movie_title}' basligiyla eslesen film bulunamadi.")
        return

    movie_id = matches['movieId'].iloc[0]
    matched_title = matches['title'].iloc[0]
    movie_idx = movie_mapper[movie_id]
    movie_vec = X[movie_idx]

    model = NearestNeighbors(metric='cosine', algorithm='brute')
    model.fit(X)
    distances, indices = model.kneighbors(movie_vec, n_neighbors=k + 1)

    neighbor_ids = [movie_inv_mapper[i] for i in indices.flatten()[1:]]
    recommendations = df[df['movieId'].isin(neighbor_ids)]['title'].unique()

    print(f"\nBecause you liked **{matched_title}**, you might also enjoy:")
    for rec in recommendations:
        print(f"- {rec}")

Adım 5: Test Edin ve Sonucu Alın

Sistemimizi test edeceğiz:

In [9]:
recommend_similar("The Dark Knight", ratings, X,
                  movie_mapper, movie_inv_mapper, k=5)


Because you liked **Dark Knight, The (2008)**, you might also enjoy:
- Inception (2010)
- Dark Knight Rises, The (2012)
- Lord of the Rings: The Return of the King, The (2003)
- Batman Begins (2005)
- Iron Man (2008)
